In [ ]:
from pathlib import Path
from datetime import datetime

import torch
from lightning.pytorch import (
    seed_everything,
)
from pytorch_lightning.loggers import WandbLogger
import wandb

from torch.utils.data import DataLoader, TensorDataset
from torchcfm.conditional_flow_matching import (
    ConditionalFlowMatcher,
)
from torch.optim.lr_scheduler import CosineAnnealingLR
from msas_pytorch import msas

from patientflow.data import BrainteaserDataModule
from patientflow.models.ae import PatientEmbeddedAE
from patientflow.models.vector_fields import MLP, TransformerVecField

In [ ]:
seed_everything(42)

In [ ]:
import warnings

warnings.simplefilter(action="ignore", category=FutureWarning)

ds = BrainteaserDataModule(
    42,
    32,
    Path.home()
    / "Research"
    / "Virtual-ALS-Patients"
    / "data"
    / "no_labels_no_birth_year",
)
ds.setup("fit")

In [ ]:
x_s = ds.train_dataset[:][0]
x_t = ds.train_dataset[:][1]
real_seq_sizes = ds.train_dataset[:][2]

In [ ]:
ae: PatientEmbeddedAE = torch.load("vae_2025-03-27 15:13:04.pt").eval()

In [ ]:
with torch.no_grad():
    encoded_static, encoded_temporal = ae.encode(x_s, x_t)
    if isinstance(encoded_static, tuple):
        encoded_static = encoded_static[0]
    if isinstance(encoded_temporal, tuple):
        encoded_temporal = encoded_temporal[0]
    recons_static, recons_temporal = ae.decode(
        encoded_static,
        encoded_temporal,
        transform_and_unify_output=True,
        adjust_to_seq_lengths=real_seq_sizes,
    )
    encoded_static = encoded_static.cpu()
    encoded_temporal = encoded_temporal.cpu()
    recons_static = recons_static.cpu()
    recons_temporal = recons_temporal.cpu()

In [ ]:
print("Min static", encoded_static.min())
print("Mean static", encoded_static.mean())
print("Std static", encoded_static.std(dim=0).mean())
print("Max static", encoded_static.max())
print("Min temporal", encoded_temporal.min())
print("Mean temporal", encoded_temporal.mean())
print("Std temporal", encoded_temporal.std(dim=0).mean())
print("Max temporal", encoded_temporal.max())

In [ ]:
discrete_temporal_features_indices = torch.LongTensor(
    ds.features.temporal_features().categorical_features_indices()
)
discrete_temporal_features_num_categories = torch.LongTensor(
    ds.features.temporal_features().categorical_features_num_categories()
)
discrete_static_features_indices = torch.LongTensor(
    ds.features.static_features().categorical_features_indices()
)
discrete_static_features_num_categories = torch.LongTensor(
    ds.features.static_features().categorical_features_num_categories()
)
msas_score_no_reduction = msas(
    x_t.cpu(),
    recons_temporal.cpu(),
    [torch.mean, torch.std, torch.median, torch.max, torch.min],
    discrete_temporal_features_indices=discrete_temporal_features_indices,
    discrete_temporal_features_num_categories=discrete_temporal_features_num_categories,
    real_static_data=x_s.cpu(),
    synthetic_static_data=recons_static.cpu(),
    discrete_static_features_indices=discrete_static_features_indices,
    discrete_static_features_num_categories=discrete_static_features_num_categories,
    enforce_temporal_shape=False,
    reduction=None,
)
msas_score = msas(
    x_t.cpu(),
    x_t.cpu(),
    [torch.mean, torch.std, torch.median, torch.max, torch.min],
    discrete_temporal_features_indices=discrete_temporal_features_indices,
    discrete_temporal_features_num_categories=discrete_temporal_features_num_categories,
    real_static_data=x_s.cpu(),
    synthetic_static_data=x_s.cpu(),
    discrete_static_features_indices=discrete_static_features_indices,
    discrete_static_features_num_categories=discrete_static_features_num_categories,
    enforce_temporal_shape=False,
)
print(f"MSAS score with no reduction: {msas_score_no_reduction}")
print(f"MSAS score: {msas_score}")

In [ ]:
discrete_temporal_features_indices = torch.LongTensor(
    ds.features.temporal_features().categorical_features_indices()
)
discrete_temporal_features_num_categories = torch.LongTensor(
    ds.features.temporal_features().categorical_features_num_categories()
)
discrete_static_features_indices = torch.LongTensor(
    ds.features.static_features().categorical_features_indices()
)
discrete_static_features_num_categories = torch.LongTensor(
    ds.features.static_features().categorical_features_num_categories()
)
msas_score_no_reduction = msas(
    x_t.cpu(),
    recons_temporal.cpu(),
    [torch.mean, torch.std, torch.median, torch.max, torch.min],
    discrete_temporal_features_indices=discrete_temporal_features_indices,
    discrete_temporal_features_num_categories=discrete_temporal_features_num_categories,
    real_static_data=x_s.cpu(),
    synthetic_static_data=recons_static.cpu(),
    discrete_static_features_indices=discrete_static_features_indices,
    discrete_static_features_num_categories=discrete_static_features_num_categories,
    enforce_temporal_shape=False,
    reduction=None,
)
msas_score = msas(
    x_t.cpu(),
    recons_temporal.cpu(),
    [torch.mean, torch.std, torch.median, torch.max, torch.min],
    discrete_temporal_features_indices=discrete_temporal_features_indices,
    discrete_temporal_features_num_categories=discrete_temporal_features_num_categories,
    real_static_data=x_s.cpu(),
    synthetic_static_data=recons_static.cpu(),
    discrete_static_features_indices=discrete_static_features_indices,
    discrete_static_features_num_categories=discrete_static_features_num_categories,
    enforce_temporal_shape=False,
)
print(f"MSAS score with no reduction: {msas_score_no_reduction}")
print(f"MSAS score: {msas_score}")

In [ ]:
device = torch.device("cuda")
n_epochs_static = 10000
n_epochs_temporal = 500

loader_static = DataLoader(TensorDataset(encoded_static), batch_size=32, shuffle=True)
loader_temporal = DataLoader(
    TensorDataset(encoded_static, encoded_temporal, real_seq_sizes),
    batch_size=16,
    shuffle=True,
)

In [ ]:
hyperparams_static = dict(
    hidden_dim=1024,
    num_hidden_layers=3,
    time_varying=True,
    dropout=0.1,
)

In [ ]:
now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
logger = WandbLogger(project="patient_flow", name=f"FlowMatching_{now}")

In [ ]:
net_static = MLP(
    encoded_static.size(1), out_dim=encoded_static.size(1), **hyperparams_static
).to(device)
hyperparams_static["lr"] = 1e-3
hyperparams_static["grad_clip"] = 0.0
hyperparams_static["batch_size"] = 32

if hyperparams_static["grad_clip"] > 0:
    torch.nn.utils.clip_grad_norm_(
        net_static.parameters(), max_norm=hyperparams_static["grad_clip"]
    )

optimizer_static = torch.optim.AdamW(
    net_static.parameters(), lr=hyperparams_static["lr"]
)
scheduler_static = CosineAnnealingLR(
    optimizer_static, T_max=n_epochs_static, eta_min=1e-5
)

In [ ]:
print(net_static)

In [ ]:
fm = ConditionalFlowMatcher(sigma=0.0)

In [ ]:
hyperparams_temporal = dict(
    transformer_encoder_n_heads=8,
    transformer_encoder_dim_forward=1024,
    transformer_encoding_n_layers=12,
    learned_pe=False,
    learned_time_embedding=False,
    time_embedding_size=32,
)

In [ ]:
logger.log_hyperparams(hyperparams_static)
logger.log_hyperparams(hyperparams_temporal)

In [ ]:
net_temporal = TransformerVecField(
    input_dim=encoded_temporal.size(2),
    conditional_static_dim=encoded_static.size(1),
    **hyperparams_temporal,
).to(device)

hyperparams_temporal["lr"] = 1e-3
hyperparams_temporal["grad_clip"] = 0.0
hyperparams_temporal["batch_size"] = 16

if hyperparams_temporal["grad_clip"] > 0:
    torch.nn.utils.clip_grad_norm_(
        net_temporal.parameters(), max_norm=hyperparams_temporal["grad_clip"]
    )

optimizer_temporal = torch.optim.AdamW(
    net_temporal.parameters(), lr=hyperparams_temporal["lr"]
)
scheduler_temporal = CosineAnnealingLR(
    optimizer_temporal, T_max=n_epochs_temporal, eta_min=1e-5
)

In [ ]:
for epoch in range(n_epochs_static):
    for i, data in enumerate(loader_static):
        b_static = data[0]
        optimizer_static.zero_grad()

        x1_static = b_static.to(device)
        x0 = torch.randn_like(x1_static).to(device)
        t, xt, ut = fm.sample_location_and_conditional_flow(x0, x1_static)
        vt_static = net_static(torch.concat((xt, t.unsqueeze(1)), dim=1))
        loss_static = torch.mean((vt_static - ut) ** 2)

        loss_static.backward()
        logger.log_metrics(
            {"train_static_loss": loss_static.item()},
            step=epoch * len(loader_static) + i,
        )
        optimizer_static.step()
        print(f"epoch: {epoch}, steps: {i}, loss: {loss_static.item():.4}", end="\r")
    scheduler_static.step()

In [ ]:
for epoch in range(n_epochs_temporal):
    for i, data in enumerate(loader_temporal):
        b_static, b_temporal, b_seq_sizes = data
        optimizer_temporal.zero_grad()

        x1_temporal = b_temporal.to(device)
        x0 = torch.randn_like(x1_temporal).to(device)
        t, xt, ut = fm.sample_location_and_conditional_flow(x0, x1_temporal)

        vt_temporal = net_temporal(
            t, xt, b_static.to(device), seq_sizes=b_seq_sizes.to(device)
        )

        # Create mask where each row has 1s up to its sequence length and 0s after
        seq_indices = torch.arange(vt_temporal.size(1), device=device).unsqueeze(0)
        mask = (seq_indices < b_seq_sizes.unsqueeze(1).to(device)).unsqueeze(-1)

        # # Apply mask to zero out padding
        ut = ut * mask
        vt_temporal = vt_temporal * mask

        loss_temporal = torch.mean((vt_temporal - ut) ** 2)
        logger.log_metrics(
            {"train_temporal_loss": loss_temporal.item()},
            step=epoch * len(loader_temporal) + i,
        )
        loss_temporal.backward()
        optimizer_temporal.step()
        print(f"epoch: {epoch}, steps: {i}, loss: {loss_temporal.item():.4}", end="\r")
    scheduler_temporal.step()

In [ ]:
logger.finalize("success")
wandb.finish()

In [ ]:
torch.save(net_static, f"net_static_{now}.pt")
torch.save(net_temporal, f"net_temporal_{now}.pt")